# CWRU 轴承数据探索

**目的**：用公开的凯斯西储大学（CWRU）轴承振动数据，跑通「读数据 → 看图 → 提特征 → 得结论」的完整流程。

**这份 notebook 是你的第一份科研作品**，最后会沉淀成 `results/reports/01_cwru_data_report.md`。

**四类状态**：正常 / 内圈故障 / 外圈故障 / 滚动体故障

**数据准备**：把从 CWRU 官网下载的 4 个 `.mat` 文件放到 `data/raw/cwru/` 下。
文件名可能带后缀差异（例如 `OR007@6_0.mat`），**以你实际下载到的文件名为准**，在下面的 `FILES` 里改。

> 还没下载数据也能跑：第 3 节会自动切换到**合成信号**，让你先看清流程；
> 等你把真实 `.mat` 文件放进去，重跑本 notebook 就是真实结果。

## 0. 环境准备

**重要**：先确认右上角内核选择的是 `Python (phm)` 或 `.\.venv\Scripts\python.exe`。
如果不是，点右上角内核名称 → 选择 `.venv` 里的解释器。

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # 让 notebook 能 import src

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils import set_seed, setup_chinese_font, RAW_DIR
from src.data_loader import load_cwru_mat
from src.features import extract_features

set_seed(42)          # 固定随机种子，保证结果可复现
setup_chinese_font()  # 中文正常显示
%matplotlib inline

print('项目数据目录:', RAW_DIR)
print('numpy', np.__version__, '| pandas', pd.__version__)

## 1. 先确认数据在不在

**科研第一习惯**：动手分析之前，先确认文件真的存在、名字真的对。

In [ ]:
CWRU_DIR = RAW_DIR / 'cwru'
CWRU_DIR.mkdir(parents=True, exist_ok=True)

found = sorted(p.name for p in CWRU_DIR.glob('*.mat'))
print(f'数据目录: {CWRU_DIR}')
print(f'发现 {len(found)} 个 .mat 文件:')
for name in found:
    print('  -', name)
if not found:
    print('\n[提示] 还没有真实数据。请到 CWRU 官网下载 12kHz Drive End 数据放进本目录。')
    print('       下面会自动使用合成信号，先把分析流程走通。')

## 2. 读一个文件，看看它长什么样

CWRU 的 `.mat` 里变量名形如 `X118_DE_time`：
- `DE` = Drive End（驱动端加速度传感器），`FE` = Fan End，`BA` = Base
- `_time` 表示时域信号（区别于 `_RPM` 转速）

采样率：**12kHz 数据集 → fs = 12000 Hz**（48kHz 数据集则是 48000）。

In [ ]:
fs = 12000   # CWRU 12kHz 驱动端数据

if (CWRU_DIR / 'Normal_0.mat').exists():
    sig, key = load_cwru_mat(CWRU_DIR / 'Normal_0.mat')
    print('文件内变量名 :', key)
    print('信号长度     :', sig.shape, '| 数据类型:', sig.dtype)
    print('前 10 个值   :', np.round(sig[:10], 4))
    print('最大值 / 最小值: %.4f / %.4f' % (sig.max(), sig.min()))
    print('RMS          : %.4f' % np.sqrt(np.mean(sig**2)))
    print('时长         : %.3f 秒' % (len(sig) / fs))
else:
    print('[跳过] 未找到 Normal_0.mat，先看合成信号的统计量')
    sig = np.random.randn(4096) * 0.1
    key = '(合成信号)'
    print('信号长度     :', sig.shape)
    print('RMS          : %.4f' % np.sqrt(np.mean(sig**2)))

In [ ]:
t = np.arange(len(sig)) / fs

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, sig, linewidth=0.5)
ax.set_xlabel('时间 (s)')
ax.set_ylabel('加速度 (g)')
ax.set_title(f'CWRU 正常轴承 驱动端振动信号（{key}）')
plt.tight_layout()
plt.show()

## 3. 四类状态一起读进来

**如果你的文件名不一样，改下面这个字典就行。**

没有真实数据时，这里会生成四类**合成信号**：正常=白噪声，故障=在噪声上叠加周期性冲击。
合成信号不能发表任何结论，但足以让你先看清「不同状态的特征长什么样」。

In [ ]:
FILES = {
    '正常':      'Normal_0.mat',
    '内圈故障':  'IR007_0.mat',
    '外圈故障':  'OR007@6_0.mat',
    '滚动体故障': 'B007_0.mat',
}

N_SAMPLES = 4096   # 每类统一取这么长，控制变量

def make_synthetic(fault_freq, impulse_amp, seed, n=N_SAMPLES):
    """生成一段「噪声 + 周期性冲击」的合成信号，用于在无数据时演示流程。"""
    rng = np.random.default_rng(seed)
    x = rng.normal(0, 0.05, n)
    if impulse_amp > 0:
        period = int(fs / fault_freq)
        for pos in range(0, n, period):
            decay = np.exp(-np.arange(10) / 2.5)
            end = min(pos + 10, n)
            x[pos:end] += impulse_amp * decay[: end - pos]
    return x

signals, data_source = {}, {}
for label, fname in FILES.items():
    path = CWRU_DIR / fname
    if path.exists():
        s, k = load_cwru_mat(path)
        signals[label] = s[:N_SAMPLES].astype(float)
        data_source[label] = '真实'
        print(f'{label:<6} 真实数据  变量 {k:<14} 采样点 {len(s)}')
    else:
        signals[label] = None
        data_source[label] = '缺失'

if any(v is None for v in signals.values()):
    print('\n[合成模式] 缺少真实数据，生成四类合成信号演示流程')
    synth = {
        '正常':      make_synthetic(0,    0.0,  seed=1),
        '内圈故障':  make_synthetic(107,  0.8,  seed=2),
        '外圈故障':  make_synthetic(83.5, 1.2,  seed=3),
        '滚动体故障': make_synthetic(70,   0.6,  seed=4),
    }
    for label, s in synth.items():
        if signals.get(label) is None:
            signals[label] = s
            data_source[label] = '合成'
else:
    print('\n[真实模式] 四类真实数据齐全')

print('\n各状态数据来源:', data_source)

## 4. 对比四类信号的时域波形

**观察重点**：故障信号的幅值是不是更大？有没有周期性的冲击（尖峰）？

In [ ]:
fig, axes = plt.subplots(len(signals), 1, figsize=(12, 2.2 * len(signals)), sharex=True)
for ax, (label, s) in zip(axes, signals.items()):
    ax.plot(np.arange(len(s)) / fs, s, linewidth=0.5)
    ax.set_ylabel(f'{label}\n({data_source[label]})')
    ax.grid(alpha=0.3)
axes[-1].set_xlabel('时间 (s)')
fig.suptitle('CWRU 四类轴承状态振动信号对比')
plt.tight_layout()
plt.show()

## 5. 提取特征，做成一张对比表

我们调用了 `src/features.py` 里的 `extract_features()`，它算了 10 个时域特征 + 4 个频域特征。

**为什么要统一取 4096 点而不是用全部？**
因为不同记录的采样点数可能不同，直接比总量不公平；统一长度才叫**控制变量**。

In [ ]:
rows = []
for label, s in signals.items():
    feats = extract_features(s, fs)
    feats['状态'] = label
    feats['数据来源'] = data_source[label]
    rows.append(feats)

df = pd.DataFrame(rows).set_index('状态')
df.round(4)

## 6. 哪个特征最能区分故障？

- **RMS（均方根）**：反映信号总能量 → 对整体幅值上升敏感
- **峭度 kurtosis**：反映分布的「尖锐程度」→ 对**冲击成分**敏感（轴承故障的典型表现）
- **峰值因子 crest_factor = 峰值 / RMS**：冲击越强，这个值越大
- **重心频率 FC**：频谱能量集中在哪个频段 → 故障会让高频能量上升
- **频谱熵**：谱的「混乱程度」→ 越接近白噪声越大

In [ ]:
for col in ['rms', 'kurtosis', 'crest_factor', 'FC', 'spectral_entropy']:
    print(f'--- {col} ---')
    print(df[col].round(4).sort_values().to_string())
    print()

In [ ]:
plot_cols = ['rms', 'kurtosis', 'crest_factor', 'FC']
fig, axes = plt.subplots(1, len(plot_cols), figsize=(4 * len(plot_cols), 3.5))
for ax, col in zip(axes, plot_cols):
    df[col].plot(kind='bar', ax=ax, title=col, rot=30)
    ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 7. 引出问题：光看时域够不够？

**请你先自己观察再往下看。**

时域特征告诉我们「有没有异常」，但很难告诉我们「是哪种异常」。
内圈故障和外圈故障的时域统计量可能很接近，但它们的**故障特征频率不同**——
这必须靠**频域（FFT）和包络谱**才能区分。

这就是你第 3 周要学信号处理的动机。带着这个问题去学，比干啃教材快 10 倍。

## 8. 要写进报告里的 5 个问题

1. 四类信号的幅值有明显差别吗？RMS 谁最大、谁最小？
2. 峭度（kurtosis）对哪种故障最敏感？你认为原因是什么？
3. 峰值因子和 RMS 反映的是同一件事吗？如果不是，区别在哪？
4. 光看时域波形或时域特征，你能区分内圈和外圈故障吗？如果不能，需要什么工具？
5. 如果只有一个特征可用，你会选哪个？为什么？它的局限是什么？

**把这 5 个问题的答案写进 notebook 的 Markdown cell，然后整理到 `results/reports/01_cwru_data_report.md`。**